[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sampriti-sr/MLP_Course/blob/kshitij-variant/docs/toolkit/numpy-vectorisation.ipynb)

# NumPy: Vectorisation, Broadcasting and Linear Algebra

<div class="admonition info">
<p class="admonition-title">What you will be able to do after this page</p>
<ul>
<li>Predict the result of a broadcast and fix a shape error.</li>
<li>Aggregate along an axis, with keepdims when you need to broadcast back.</li>
<li>Compute all pairwise distances without a loop.</li>
<li>Solve least squares robustly and explain why collinearity breaks the textbook formula.</li>
</ul>
</div>

**Prerequisites:** <a href="../numpy-arrays/">NumPy: Arrays and Indexing</a>. **Dataset:** Online Retail II. **Time:** about 40 minutes.

In [1]:
try:
    import mlp_data
except ImportError:  # e.g. a fresh Colab runtime
    %pip install -q "mlp-course @ git+https://github.com/sampriti-sr/MLP_Course@kshitij-variant"
    import mlp_data

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlp_data import load_retail, retail, style

style.apply()
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 14)
pd.set_option("display.float_format", "{:,.2f}".format)

## The situation

The previous page was about *holding* data in arrays. This one is about *computing* with them: arithmetic between arrays of different shapes (**broadcasting**), statistics along an axis, distances between every pair of points, and linear algebra. These four ideas underlie nearly every algorithm you will meet: standardisation, k-means, regression, PCA.

We use a real matrix: three behavioural numbers (recency, frequency, spend) for each of the 4,324 active customers in the retail case study.

In [2]:
cleaned = retail.clean_retail(load_retail())
customers = retail.build_splits(cleaned.sales, cleaned.returns).valid
X = customers[["recency_days", "frequency", "monetary"]].to_numpy(copy=True)       # one row per customer, one column per feature
print("shape:", X.shape, "| dtype:", X.dtype)
pd.DataFrame(X[:4], columns=["recency_days", "frequency", "monetary"])

shape: (4324, 3) | dtype: float64


,recency_days,frequency,monetary
0,337.42,1.00,142.31
1,54.55,4.00,"2,434.96"
2,56.55,4.00,"1,388.40"
3,215.65,1.00,"1,152.62"


## 1. Broadcasting: arithmetic between different shapes

When two arrays have different shapes, NumPy tries to **stretch** the smaller one to fit. The rule is applied from the **trailing (rightmost) dimension** backwards: two dimensions are compatible if they are equal, or if one of them is 1.

| `A` shape | `B` shape | Result | Why |
|---|---|---|---|
| (4324, 3) | (3,) | (4324, 3) | the row of 3 is repeated down all 4,324 rows |
| (4324, 3) | (4324, 1) | (4324, 3) | the column is repeated across the 3 columns |
| (4324, 3) | (4324,) | **error** | trailing dimensions 3 and 4,324 do not match |

The most common use is standardising each column: subtract that column's mean and divide by its standard deviation. `X.mean(axis=0)` collapses the rows and leaves one value per column, shape `(3,)`, which broadcasts back across every row.

In [3]:
means, stds = X.mean(axis=0), X.std(axis=0)
print("column means:", means.round(1), " shapes:", means.shape, "->", (X - means).shape)

Z = (X - means) / stds                                   # standardise every column at once, no loop
print("after standardising, column means:", Z.mean(axis=0).round(6), " column std:", Z.std(axis=0).round(3))

column means: [ 130.5    4.1 1839.9]  shapes: (3,) -> (4324, 3)
after standardising, column means: [-0. -0. -0.]  column std: [1. 1. 1.]


**The shape error is the one to recognise.** Suppose you wanted to subtract each customer's *recency* (a single value per row) from all three of their columns. The recency column has shape `(4324,)`, which lines up with the *wrong* end:

In [4]:
recency = X[:, 0]
try:
    X - recency
except ValueError as e:
    print("ValueError:", e)

fixed = X - recency[:, None]                             # [:, None] turns (4324,) into (4324, 1): a column
print("fixed shape:", fixed.shape, "| first column is now all zeros:", bool((fixed[:, 0] == 0).all()))

ValueError: operands could not be broadcast together with shapes (4324,3) (4324,) 
fixed shape: (4324, 3) | first column is now all zeros: True


`[:, None]` (or `np.newaxis`, or `.reshape(-1, 1)`) adds a length-1 axis, so the vector stretches across columns instead of down rows. **When you see "operands could not be broadcast together", write down both shapes and line them up from the right.**

## 2. Statistics along an axis

The same `axis` rule as before: the axis you name is the one that disappears.

In [5]:
summary = pd.DataFrame({"median": np.median(X, axis=0), "90th percentile": np.percentile(X, 90, axis=0), "maximum": X.max(axis=0)},
                       index=["recency_days", "frequency", "monetary"])
print(summary.map("{:,.1f}".format).to_string())

corr = np.corrcoef(X, rowvar=False)                      # rowvar=False: columns are the variables
print("\ncorrelation matrix:\n", pd.DataFrame(corr, index=summary.index, columns=summary.index).round(2).to_string())

             median 90th percentile    maximum
recency_days  110.4           270.5      364.5
frequency       2.0             8.0      177.0
monetary      642.9         3,447.3  252,425.5

correlation matrix:
               recency_days  frequency  monetary
recency_days          1.00      -0.33     -0.18
frequency            -0.33       1.00      0.61
monetary             -0.18       0.61      1.00


The spend column has a maximum many times its 90th percentile (the heavy tail we met in the lifecycle chapter), and frequency and spend are clearly correlated (0.61 on the raw values; the heavy tail dilutes this linear measure, and the rank correlation used earlier is higher). `np.corrcoef` expects **variables in rows** by default, so pass `rowvar=False` for a typical (samples × features) matrix.

## 3. Distances between every pair, by broadcasting

Clustering algorithms like k-means repeatedly ask "which centre is each customer nearest to?" That is a distance between every customer and every centre, with no loop. Give the customers shape `(n, 1, 3)` and the centres shape `(1, k, 3)`; broadcasting produces all `n × k` differences at once.

In [6]:
rng = np.random.default_rng(0)                           # a seeded generator: reproducible
centres = Z[rng.choice(len(Z), size=4, replace=False)]   # four customers, picked at random, as starting centres

diff = Z[:, None, :] - centres[None, :, :]               # (4324, 1, 3) - (1, 4, 3) -> (4324, 4, 3)
sq_dist = (diff ** 2).sum(axis=2)                        # squared Euclidean distance: (4324, 4)
nearest = sq_dist.argmin(axis=1)                         # the closest centre for each customer

print("intermediate array:", diff.shape, f"= {diff.nbytes / 1e6:.1f} MB;  distances:", sq_dist.shape)
print("customers per centre:", np.bincount(nearest, minlength=4))

intermediate array: (4324, 4, 3) = 0.4 MB;  distances: (4324, 4)
customers per centre: [ 505 2457  651  711]


Notice the cost: the intermediate array is `n × k × d`, which for millions of rows and many centres becomes large. Libraries use a smarter formula for big problems, but the broadcasting version is how you *understand* it.

## 4. Linear algebra: least squares three ways

Fit a straight line (a plane, here) predicting log-spend from log-frequency and log-recency.

<div class="admonition math">
<p class="admonition-title">Maths ↔ code</p>
<p>Ordinary least squares finds the coefficients <code>β</code> minimising the squared error, and the textbook answer is <code>β = (XᵀX)⁻¹ Xᵀy</code>. In NumPy: <code>np.linalg.solve(X.T @ X, X.T @ y)</code>. Note <code>@</code> is matrix multiplication and <code>.T</code> is the transpose. (We solve the system rather than computing the inverse, which is slower and less accurate.)</p>
</div>

In [7]:
y = np.log1p(customers["monetary"].to_numpy())
A = np.column_stack([np.ones(len(customers)), np.log1p(customers["frequency"]), np.log1p(customers["recency_days"])])   # design matrix with an intercept column

beta_normal = np.linalg.solve(A.T @ A, A.T @ y)          # the textbook formula
beta_lstsq = np.linalg.lstsq(A, y, rcond=None)[0]        # a numerically safer routine
from sklearn.linear_model import LinearRegression
sk = LinearRegression().fit(A[:, 1:], y)                 # scikit-learn adds its own intercept

print("normal equations:", beta_normal.round(4))
print("lstsq:           ", beta_lstsq.round(4))
print("scikit-learn:    ", np.r_[sk.intercept_, sk.coef_].round(4))
assert np.allclose(beta_normal, beta_lstsq) and np.allclose(beta_normal, np.r_[sk.intercept_, sk.coef_])

residual = y - A @ beta_lstsq
print(f"R-squared: {1 - (residual ** 2).sum() / ((y - y.mean()) ** 2).sum():.3f}")

normal equations: [ 4.7063  1.484  -0.0281]
lstsq:            [ 4.7063  1.484  -0.0281]
scikit-learn:     [ 4.7063  1.484  -0.0281]
R-squared: 0.659


Three routes, one answer. Why prefer `lstsq`? Because forming `XᵀX` **squares the condition number**, a measure of how sensitive the answer is to small errors:

In [8]:
print(f"condition number of A:       {np.linalg.cond(A):8.1f}")
print(f"condition number of A.T @ A:   {np.linalg.cond(A.T @ A):8.1f}   (about the square of the one above)")

A_dup = np.column_stack([A, 2 * A[:, 1]])                # add a column that is exactly twice another: perfect collinearity
try:
    np.linalg.solve(A_dup.T @ A_dup, A_dup.T @ y)
    print("solve: returned an answer")
except np.linalg.LinAlgError as e:
    print("solve fails:", e)
print("lstsq still answers; its rank detection says the matrix has rank", np.linalg.lstsq(A_dup, y, rcond=None)[2], "not", A_dup.shape[1])

condition number of A:           30.6
condition number of A.T @ A:      938.6   (about the square of the one above)
solve fails: Singular matrix
lstsq still answers; its rank detection says the matrix has rank 3 not 4


With a perfectly redundant column the normal equations have no unique solution and `solve` gives up (or, with near-collinearity, silently returns huge unstable numbers), while `lstsq` reports the true **rank** and returns a sensible answer. This is the same phenomenon that made single coefficients meaningless on the interpretation page: correlated inputs make least squares ill-conditioned.

## 5. Random numbers you can reproduce

Use a **seeded generator**, `np.random.default_rng(seed)`, and pass it around. The same seed gives the same numbers every time, which is what makes experiments repeatable. A vectorised **bootstrap** shows the style: draw a thousand resamples of the customers at once as one `(1000, n)` array of row indices.

In [9]:
rng = np.random.default_rng(42)
spend = customers["monetary"].to_numpy()
picks = rng.integers(0, len(spend), size=(1000, len(spend)))     # 1000 resamples, each the same size as the data
boot_means = spend[picks].mean(axis=1)                           # the mean of each resample: shape (1000,)

low, high = np.percentile(boot_means, [5, 95])
print(f"mean spend {spend.mean():,.0f}; bootstrap 90% interval {low:,.0f} to {high:,.0f}")

again = np.random.default_rng(42).integers(0, len(spend), size=5)
print("same seed, same numbers:", np.array_equal(again, np.random.default_rng(42).integers(0, len(spend), size=5)))

mean spend 1,840; bootstrap 90% interval 1,659 to 2,025
same seed, same numbers: True


That is a thousand resampled means computed without a Python loop. (The interval is about ten percent either side of the mean, which sounds tight, but remember that this mean is dominated by a few very large customers: the median would tell a very different story.)

## Pitfalls at a glance

| Surprise | Cause | Fix |
|---|---|---|
| "operands could not be broadcast together" | shapes do not line up from the right | print both shapes; add an axis with `[:, None]` |
| Standardising down the wrong axis | `axis=0` collapses rows (per-column stats); `axis=1` per-row | name the axis you want to *disappear* |
| `corrcoef` result looks wrong | it expects variables in rows | `rowvar=False` |
| Huge memory in a pairwise-distance step | the `(n, k, d)` intermediate | chunk the rows, or use a library routine |
| Wild regression coefficients | collinear columns make `XᵀX` ill-conditioned | `lstsq`, regularisation, drop redundant columns |
| Different results every run | an unseeded generator | `np.random.default_rng(seed)` |

## Check yourself

<details class="question">
<summary>1. <code>A</code> has shape (5, 4) and <code>b</code> has shape (5,). Why does <code>A - b</code> fail, and how do you subtract <code>b</code> from every column?</summary>

Broadcasting matches shapes from the right: `(5, 4)` against `(5,)` compares 4 with 5, which do not match. Make `b` a column, `b[:, None]` with shape `(5, 1)`, and it stretches across the 4 columns.

</details>

<details class="question">
<summary>2. What are the shapes of <code>Z[:, None, :]</code> and <code>centres[None, :, :]</code>, and why does subtracting them give every customer-centre pair?</summary>

`(4324, 1, 3)` and `(1, 4, 3)`. The length-1 axes stretch: the customers' singleton axis grows to 4 and the centres' to 4324, giving `(4324, 4, 3)`, one 3-vector difference for each (customer, centre) pair.

</details>

<details class="question">
<summary>3. Why prefer <code>np.linalg.lstsq</code> to <code>solve(X.T @ X, X.T @ y)</code>?</summary>

Forming `XᵀX` squares the condition number, magnifying numerical error, and the normal equations fail outright when columns are perfectly redundant. `lstsq` works on `X` directly (using orthogonal factorisations) and reports the rank, so it is more accurate and more graceful.

</details>

## Exercises

<details class="question">
<summary>Write a <code>softmax</code> that turns each row of a matrix into probabilities (non-negative, summing to 1). Test it on <code>Z[:5]</code>. What goes wrong if you forget <code>keepdims=True</code>?</summary>

```python
def softmax(M):
    shifted = M - M.max(axis=1, keepdims=True)            # subtract the row max for numerical stability
    e = np.exp(shifted)
    return e / e.sum(axis=1, keepdims=True)

P = softmax(Z[:5])
print(P.sum(axis=1))                                      # each row sums to 1
```

Without `keepdims=True`, `M.max(axis=1)` has shape `(5,)`, and subtracting it from a `(5, 3)` matrix raises the broadcast error from section 1. `keepdims=True` keeps the reduced axis as length 1, giving `(5, 1)`, which stretches correctly.

</details>

<details class="question">
<summary>Reassign every customer to its nearest centre using <em>Manhattan</em> distance (the sum of absolute differences) instead of squared Euclidean. Do the group sizes change?</summary>

```python
manhattan = np.abs(Z[:, None, :] - centres[None, :, :]).sum(axis=2)
print(np.bincount(manhattan.argmin(axis=1), minlength=4))
```

Only the line that turns differences into a distance changes (`np.abs(...)` instead of `** 2`). The group sizes usually shift somewhat, because a different distance defines "nearest" differently, especially for the heavy-tailed customers far from every centre.

</details>

## Recap

- **Broadcasting** stretches the smaller array; compare shapes from the **right**, and use `[:, None]` to add an axis.
- `axis` names the dimension that **disappears**; `keepdims=True` keeps it as length 1 so results broadcast back.
- Pairwise distances come from broadcasting `(n, 1, d)` against `(1, k, d)`, at the cost of an `(n, k, d)` intermediate.
- Least squares: `lstsq` beats the normal equations, which square the condition number and fail on collinear columns.
- Seed a generator (`default_rng`) and vectorise resampling.

**Next:** the pandas pages, which wrap this machinery in labelled tables.